# Talus-4 Training on Google Colab Pro (NVIDIA L4 24 GB)

This notebook builds the `earth64` dataset (Copernicus GLO-30 real terrain, ~50k 64×64 windows at 64 m/cell) once onto Google Drive, warm-starts from **Talus-3**, and trains **Talus-4** on an L4 GPU with automatic checkpoint recovery if a Colab session disconnects.

**Before running:** In Colab, select **Runtime → Change runtime type → Hardware accelerator: L4 GPU**.

In [ ]:
# 1. Verify GPU and mount Google Drive for persistent runs & cached dataset
!nvidia-smi
import os
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA GPU not found — select Runtime -> Change runtime type -> L4 GPU"
print(f"PyTorch {torch.__version__}, GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB)")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/nullscape")
(DRIVE_ROOT / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "datasets").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "releases").mkdir(parents=True, exist_ok=True)

# Keep datasets and DEM tile cache on fast local NVMe (/content), and save runs/checkpoints directly to Google Drive
os.environ["NULLSCAPE_DATA_ROOT"] = "/content/data"
os.environ["NULLSCAPE_DEM_CACHE"] = "/content/dem_cache"
os.environ["NULLSCAPE_RUNS_ROOT"] = str(DRIVE_ROOT / "runs")
os.environ["PYTHONUNBUFFERED"] = "1"
print("Drive storage ready at:", DRIVE_ROOT)

In [ ]:
# 2. Clone / update repository and install nullscape
REPO_DIR = Path("/content/talus")
if not REPO_DIR.exists():
    !git clone https://github.com/osfv/talus.git /content/talus
else:
    !git -C /content/talus pull --ff-only

%cd /content/talus
!pip install -q -U pip setuptools wheel
!pip install -q -e ".[earth,web]"

In [ ]:
# 3. Download Talus-3 release checkpoint (for warm-starting Talus-4)
import hashlib
import urllib.request

CKPT_PATH = Path("/content/talus/checkpoints/talus-3.pt")
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
TALUS3_URL = "https://github.com/osfv/talus/releases/download/v3.0.0/talus-3.pt"
TALUS3_SHA256_PREFIX = "13b4297b"

if not CKPT_PATH.is_file():
    drive_ckpt = DRIVE_ROOT / "releases" / "talus-3.pt"
    if drive_ckpt.is_file():
        !cp "{drive_ckpt}" "{CKPT_PATH}"
    else:
        print("Downloading talus-3.pt from GitHub Releases...")
        urllib.request.urlretrieve(TALUS3_URL, CKPT_PATH)
        !cp "{CKPT_PATH}" "{drive_ckpt}"

sha = hashlib.file_digest(CKPT_PATH.open("rb"), "sha256").hexdigest()
assert sha.startswith(TALUS3_SHA256_PREFIX), f"Unexpected sha256: {sha}"
print(f"talus-3.pt ready ({CKPT_PATH.stat().st_size / 1e6:.1f} MB, sha256={sha[:16]}...)")

In [ ]:
# 4. Restore earth64 from Google Drive (3s) OR build it once from AWS Copernicus DEM and archive to Drive
LOCAL_EARTH64 = Path("/content/data/earth64")
DRIVE_ARCHIVE = DRIVE_ROOT / "datasets" / "earth64.tar"

if (LOCAL_EARTH64 / "manifest.json").is_file():
    print("earth64 is already unpacked on local SSD:", LOCAL_EARTH64)
elif DRIVE_ARCHIVE.is_file():
    print("Restoring earth64 from Google Drive archive to local SSD...")
    Path("/content/data").mkdir(parents=True, exist_ok=True)
    !tar -xf "{DRIVE_ARCHIVE}" -C /content/data
    print("Restored:", LOCAL_EARTH64)
else:
    print("Building earth64 from Copernicus GLO-30 on AWS S3 (first time only; archives ~400 MB result to Drive)...")
    !nullscape earth build --config configs/dataset/earth64.yaml --max-gb 75 --workers 4
    !tar -cf "{DRIVE_ARCHIVE}" -C /content/data earth64
    print("Archived built dataset to Google Drive:", DRIVE_ARCHIVE)
    # Free the 59 GB raw DEM tile cache from local disk now that earth64 is built and archived
    !rm -rf /content/dem_cache

In [ ]:
# 5. GPU Preflight: 3 forward/backward/AdamW/EMA steps on earth64 (verifies VRAM and warm-start before writing a run)
!nullscape train --config configs/train/talus4_earth64.yaml --dry-run

In [ ]:
# 6. Train Talus-4 (automatically resumes from the latest checkpoint on Google Drive if a session was interrupted)
import json

CONFIG = "configs/train/talus4_earth64.yaml"
runs_dir = DRIVE_ROOT / "runs"
resume_ckpt = None

for folder in sorted(runs_dir.glob("*_talus4_earth64*"), reverse=True):
    status_file = folder / "status.json"
    last_pt = folder / "checkpoints" / "last.pt"
    if status_file.is_file() and last_pt.is_file():
        st = json.loads(status_file.read_text())
        if st.get("state") != "completed" and st.get("step", 0) < st.get("max_steps", 20000):
            resume_ckpt = last_pt
            print(f"Found incomplete run at step {st['step']}/{st['max_steps']}: resuming from {resume_ckpt}")
            break

if resume_ckpt is not None:
    !nullscape train --config {CONFIG} --resume "{resume_ckpt}"
else:
    !nullscape train --config {CONFIG}

In [ ]:
# 7. Evaluate the finished Talus-4 checkpoint on VAL and export release + web ONNX files to Google Drive
latest_run = max((DRIVE_ROOT / "runs").glob("*_talus4_earth64*"), key=lambda p: p.stat().st_mtime)
last_ckpt = latest_run / "checkpoints" / "last.pt"
print("Evaluating:", last_ckpt)

eval_out = DRIVE_ROOT / "evaluations" / latest_run.name
if not eval_out.exists():
    !nullscape evaluate --checkpoint "{last_ckpt}" --dataset earth64 --n 500 --repeat-seeds 0,1,2 --out "{eval_out}"

release_pt = DRIVE_ROOT / "releases" / "talus-4.pt"
if not release_pt.exists():
    !nullscape release-checkpoint --checkpoint "{last_ckpt}" --out "{release_pt}" --name Talus-4 --version 4.0.0

web_out = DRIVE_ROOT / "releases" / "web_talus4"
!python scripts/export_web.py --checkpoint "{release_pt}" --out "{web_out}" --name talus-4
print("All artifacts saved to Google Drive:", DRIVE_ROOT)